<a href="https://colab.research.google.com/github/Aggrakurnia/Triple_AI/blob/main/ML_Mahasiswa_Lulus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
import os
os.chdir('/content/drive/MyDrive/Tugas_Machine_Learning_1')
print("Working directory:", os.getcwd())
print("Isi folder:", os.listdir())

Working directory: /content/drive/MyDrive/Tugas_Machine_Learning_1
Isi folder: ['mahasiswa_lulus.csv', 'hasil_evaluasi_model.csv', 'confusion_matrices.png', 'decision_tree_plot.png', 'perbandingan_metrik.png', 'feature_importance.png', 'contoh_error_prediksi.csv']


In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

RANDOM_STATE = 42

# -----------------------------------------------------------------
# 1. LOAD DATA
# -----------------------------------------------------------------
df = pd.read_csv("mahasiswa_lulus.csv")
print("=" * 65)
print("1. DATA MENTAH")
print("=" * 65)
print(f"Jumlah baris x kolom : {df.shape}")
print("\nInfo tipe data & missing value:")
print(df.isna().sum())
print("\nContoh 5 baris pertama:")
print(df.head())

# -----------------------------------------------------------------
# 2. PREPROCESSING
# -----------------------------------------------------------------
print("\n" + "=" * 65)
print("2. PREPROCESSING")
print("=" * 65)

df_clean = df.copy()

# 2a. Handle missing value
# - Numerik (IPK, Kehadiran, Jam_Belajar)
num_cols = ["IPK", "Kehadiran", "Jam_Belajar"]
for col in num_cols:
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)
    print(f"Missing '{col}' diisi dengan median = {median_val}")

# - Kategorikal (Organisasi) -> isi dengan modus (nilai paling sering)
cat_missing_cols = ["Organisasi"]
for col in cat_missing_cols:
    mode_val = df_clean[col].mode()[0]
    df_clean[col] = df_clean[col].fillna(mode_val)
    print(f"Missing '{col}' diisi dengan modus = {mode_val}")

print(f"\nTotal missing value setelah dibersihkan: {df_clean.isna().sum().sum()}")

# 2b. Encoding fitur kategorikal
label_encoders = {}
cat_cols = ["Organisasi", "Penghasilan_Ortu", "Jenis_Kelamin", "Status_Beasiswa"]
for col in cat_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col])
    label_encoders[col] = le
    print(f"Encoding '{col}': {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Encoding target
target_le = LabelEncoder()
df_clean["Lulus_Tepat_Waktu"] = target_le.fit_transform(df_clean["Lulus_Tepat_Waktu"])
print(f"Encoding target 'Lulus_Tepat_Waktu': "
      f"{dict(zip(target_le.classes_, target_le.transform(target_le.classes_)))}")

# 2c. Pisahkan fitur (X) dan target (y)
X = df_clean.drop(columns=["Lulus_Tepat_Waktu"])
y = df_clean["Lulus_Tepat_Waktu"]

# 2d. Train-test split 80:20 (stratify agar proporsi kelas terjaga)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"\nJumlah data latih : {X_train.shape[0]}")
print(f"Jumlah data uji    : {X_test.shape[0]}")

# -----------------------------------------------------------------
# 3. MODELING
# -----------------------------------------------------------------
print("\n" + "=" * 65)
print("3. MODELING")
print("=" * 65)

models = {
    "Decision Tree (max_depth=3)": DecisionTreeClassifier(
        max_depth=3, random_state=RANDOM_STATE
    ),
    "Decision Tree (max_depth=None)": DecisionTreeClassifier(
        max_depth=None, random_state=RANDOM_STATE
    ),
    "Gaussian Naive Bayes": GaussianNB(),
}

trained_models = {}
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    trained_models[name] = model
    predictions[name] = y_pred
    print(f"Model '{name}' selesai dilatih.")

# -----------------------------------------------------------------
# 4. EVALUASI
# -----------------------------------------------------------------
print("\n" + "=" * 65)
print("4. EVALUASI MODEL")
print("=" * 65)

results = []
for name, y_pred in predictions.items():
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    results.append({
        "Model": name, "Accuracy": acc, "Precision": prec,
        "Recall": rec, "F1-Score": f1
    })
    print(f"\n--- {name} ---")
    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1-Score  : {f1:.4f}")
    print("Classification Report:")
    print(classification_report(y_test, y_pred, target_names=target_le.classes_))

results_df = pd.DataFrame(results)
results_df.to_csv("hasil_evaluasi_model.csv", index=False)
print("\nTabel ringkasan metrik:")
print(results_df.round(4).to_string(index=False))

# 4a. Confusion Matrix (3 model dalam 1 figure)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_le.classes_)
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name, fontsize=10)
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.close()
print("\nConfusion matrix disimpan -> confusion_matrices.png")

# -----------------------------------------------------------------
# 5. VISUALISASI
# -----------------------------------------------------------------
print("\n" + "=" * 65)
print("5. VISUALISASI")
print("=" * 65)

# 5a. Plot pohon keputusan (max_depth=3)
plt.figure(figsize=(20, 10))
plot_tree(
    trained_models["Decision Tree (max_depth=3)"],
    feature_names=X.columns,
    class_names=target_le.classes_,
    filled=True, rounded=True, fontsize=10
)
plt.title("Decision Tree (max_depth=3) - Prediksi Lulus Tepat Waktu")
plt.savefig("decision_tree_plot.png", dpi=150, bbox_inches="tight")
plt.close()
print("Plot pohon keputusan disimpan -> decision_tree_plot.png")

# 5b. Bar chart perbandingan metrik 3 model
metrics_names = ["Accuracy", "Precision", "Recall", "F1-Score"]
x_pos = np.arange(len(metrics_names))
width = 0.25

fig, ax = plt.subplots(figsize=(11, 6))
for i, row in results_df.iterrows():
    values = [row[m] for m in metrics_names]
    ax.bar(x_pos + i * width, values, width, label=row["Model"])

ax.set_xticks(x_pos + width)
ax.set_xticklabels(metrics_names)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Skor")
ax.set_title("Perbandingan Metrik Evaluasi: Decision Tree vs Naive Bayes")
ax.legend(loc="lower right", fontsize=9)
ax.grid(axis="y", linestyle="--", alpha=0.5)
for i, row in results_df.iterrows():
    for j, m in enumerate(metrics_names):
        ax.text(x_pos[j] + i * width, row[m] + 0.01, f"{row[m]:.2f}",
                 ha="center", fontsize=7)
plt.tight_layout()
plt.savefig("perbandingan_metrik.png", dpi=150, bbox_inches="tight")
plt.close()
print("Bar chart perbandingan metrik disimpan -> perbandingan_metrik.png")

# 5c. Feature importance (Decision Tree max_depth=3) - bonus insight
plt.figure(figsize=(8, 5))
importances = trained_models["Decision Tree (max_depth=3)"].feature_importances_
order = np.argsort(importances)[::-1]
plt.bar(range(len(importances)), importances[order])
plt.xticks(range(len(importances)), X.columns[order], rotation=45, ha="right")
plt.ylabel("Feature Importance")
plt.title("Feature Importance - Decision Tree (max_depth=3)")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight")
plt.close()
print("Feature importance disimpan -> feature_importance.png")

# -----------------------------------------------------------------
# 6. ANALISIS ERROR (5 data salah prediksi)
# -----------------------------------------------------------------
print("\n" + "=" * 65)
print("6. ANALISIS ERROR (Decision Tree max_depth=3)")
print("=" * 65)

y_pred_dt3 = predictions["Decision Tree (max_depth=3)"]
X_test_original = df.loc[X_test.index]

error_mask = (y_test.values != y_pred_dt3)
error_idx = X_test.index[error_mask]

error_analysis = X_test_original.loc[error_idx].copy()
error_analysis["Prediksi"] = target_le.inverse_transform(y_pred_dt3[error_mask])
error_analysis["Aktual"] = target_le.inverse_transform(y_test.values[error_mask])

print(f"Total data salah prediksi (dari model max_depth=3): {len(error_analysis)} "
      f"dari {len(y_test)} data uji")

sample_errors = error_analysis.head(5)
sample_errors.to_csv("contoh_error_prediksi.csv", index=False)
print("\n5 contoh data yang salah prediksi:")
print(sample_errors.to_string(index=False))

print("""
Analisis kualitatif (interpretasi manual, dilampirkan di laporan):
- Banyak error terjadi pada mahasiswa dengan nilai fitur di sekitar
  garis batas (borderline), misalnya IPK ~3.0 atau Kehadiran ~75-80%,
  di mana kombinasi fitur saling bertentangan (IPK tinggi tapi
  kehadiran rendah, atau sebaliknya).
- Decision Tree dangkal (max_depth=3) hanya menggunakan sedikit
  aturan pemisah sehingga rentan salah pada kasus 'zona abu-abu' ini.
- Kemungkinan ada noise/faktor lain di luar dataset (motivasi
  pribadi, masalah keluarga, dsb.) yang tidak tertangkap fitur yang ada.
""")

# -----------------------------------------------------------------
# 7. SIMULASI 1 DATA BARU (Bagian C.3)
# -----------------------------------------------------------------
print("=" * 65)
print("7. SIMULASI: IPK TINGGI, KEHADIRAN RENDAH")
print("=" * 65)

# Buat 1 data baru: IPK tinggi (3.8), Kehadiran rendah (55%),
# nilai fitur lain diset ke rata-rata/modus data training
sim_data = pd.DataFrame({
    "IPK": [3.80],
    "Kehadiran": [55.0],
    "Jam_Belajar": [X_train["Jam_Belajar"].median()],
    "Organisasi": [X_train["Organisasi"].mode()[0]],
    "Penghasilan_Ortu": [X_train["Penghasilan_Ortu"].mode()[0]],
    "Jenis_Kelamin": [X_train["Jenis_Kelamin"].mode()[0]],
    "Status_Beasiswa": [X_train["Status_Beasiswa"].mode()[0]],
})[X.columns]

for name in ["Decision Tree (max_depth=3)", "Gaussian Naive Bayes"]:
    model = trained_models[name]
    pred = model.predict(sim_data)[0]
    proba = model.predict_proba(sim_data)[0]
    label = target_le.inverse_transform([pred])[0]
    print(f"\n{name}")
    print(f"  Prediksi : {label}")
    print(f"  Probabilitas : "
          f"{dict(zip(target_le.classes_, np.round(proba, 3)))}")

print("\nSelesai. Semua hasil (gambar & csv) tersimpan di folder kerja.")

1. DATA MENTAH
Jumlah baris x kolom : (500, 8)

Info tipe data & missing value:
IPK                  20
Kehadiran            20
Jam_Belajar          20
Organisasi           20
Penghasilan_Ortu      0
Jenis_Kelamin         0
Status_Beasiswa       0
Lulus_Tepat_Waktu     0
dtype: int64

Contoh 5 baris pertama:
    IPK  Kehadiran  Jam_Belajar   Organisasi Penghasilan_Ortu Jenis_Kelamin  \
0  3.32       93.1          6.3        Aktif         Menengah     Perempuan   
1  3.04      100.0          5.7  Tidak Aktif           Rendah     Perempuan   
2  3.39       65.2          3.1  Tidak Aktif         Menengah     Perempuan   
3  3.79       88.8          2.5        Aktif           Rendah     Laki-laki   
4  2.99       74.2          4.6        Aktif         Menengah     Laki-laki   

  Status_Beasiswa Lulus_Tepat_Waktu  
0              Ya       Tepat Waktu  
1           Tidak       Tepat Waktu  
2           Tidak       Tepat Waktu  
3              Ya       Tepat Waktu  
4           Tidak       T